# NB 43 — The Solenoid Lagrangian

NB 42 proved the algebra is separable: $\mathbb{Z}^*_{210} \cong \mathbb{Z}_2 \times \mathbb{Z}_4 \times \mathbb{Z}_6$
via CRT. Every algebraic quantity factors into per-prime contributions.
No coupling lives in the algebra.

This notebook asks: **where does coupling come from?**

The answer is **geometry**. The covering constraints
$\theta_{k-1} = p_k\,\theta_k$ tie adjacent levels together,
creating a tridiagonal stiffness matrix $K$ that is entirely
determined by $\{2,3,5,7\}$. Its normal modes give the oscillation
spectrum of the solenoid — with zero free parameters.

| Symbol | Meaning |
|--------|---------|
| $\theta_k$ | Angular position at nesting level $k$ |
| $P_k = \prod_{j \le k} p_j$ | Primorial at level $k$ (mass) |
| $K$ | Stiffness matrix from covering constraints |
| $W$ | Mass matrix $= \mathrm{diag}(P_0, \ldots, P_4)$ |
| $\omega^2$ | Squared normal-mode frequency |

In [1]:
import numpy as np
from fractions import Fraction
from scipy.linalg import eigh
import sys, os
sys.path.insert(0, os.path.join(os.pardir, "scripts"))
from solenoid_algebra import (
    SA, PRIMES, P, PHI, PRIMORIALS, PRIMORIAL_WEIGHTS,
    FACULTY_LABELS
)

primes = PRIMES                          # [2, 3, 5, 7]
n_levels = len(primes) + 1               # 5 angles
primorial_seq = [1] + list(PRIMORIALS)   # [1, 2, 6, 30, 210]
print(f"Primes: {primes}")
print(f"Primorials (mass): {primorial_seq}")
print(f"Levels: {n_levels}")

Primes: [2, 3, 5, 7]
Primorials (mass): [1, 2, 6, 30, 210]
Levels: 5


## 1 — The Covering Constraint

The solenoid has five angular coordinates $\theta_0, \ldots, \theta_4$.
The covering map $\pi_k : S^1_k \to S^1_{k-1}$ sends $\theta_k \mapsto p_k\,\theta_k$.
Perfect alignment means

$$\theta_{k-1} \;=\; p_k\,\theta_k, \qquad k = 1,\ldots,4.$$

Small deviations cost energy. The covering potential is

$$V \;=\; \frac{\kappa}{2}\sum_{k=1}^{4}\bigl(\theta_{k-1} - p_k\,\theta_k\bigr)^2
\;=\; \frac{\kappa}{2}\,\boldsymbol\theta^\top K\,\boldsymbol\theta$$

where $K$ is the **stiffness matrix** — the Hessian of $V/\kappa$.

The kinetic energy uses primorial masses (each level's inertia is
proportional to the number of sheets at that level):

$$T \;=\; \tfrac{1}{2}\,\dot{\boldsymbol\theta}^\top W\,\dot{\boldsymbol\theta},
\qquad W = \mathrm{diag}(P_0, P_1, P_2, P_3, P_4)
= \mathrm{diag}(1, 2, 6, 30, 210).$$

In [2]:
# Build stiffness matrix K from covering constraints
# V_k = (1/2)(theta_{k-1} - p_k * theta_k)^2
# => K[k-1,k-1] += 1, K[k,k] += p_k^2, K[k-1,k] = K[k,k-1] = -p_k

K = np.zeros((n_levels, n_levels))
for i, p in enumerate(primes):
    lo, hi = i, i + 1
    K[lo, lo] += 1
    K[hi, hi] += p**2
    K[lo, hi] -= p
    K[hi, lo] -= p

print("Stiffness matrix K:")
print(K.astype(int))

print("\nStructure:")
print(f"  K[0,0] = 1  (top level, one constraint)")
for i, p in enumerate(primes):
    k = i + 1
    if k < n_levels - 1:
        print(f"  K[{k},{k}] = {int(K[k,k])}  = {p}^2 + 1")
    else:
        print(f"  K[{k},{k}] = {int(K[k,k])}  = {p}^2  (bottom level)")
    print(f"  K[{k-1},{k}] = {int(K[k-1,k])}  = -{p}")

evals_K = np.linalg.eigvalsh(K)
print(f"\nEigenvalues of K: {np.array2string(evals_K, precision=4)}")
print(f"Rank = {np.sum(evals_K > 1e-10)}  (one zero mode)")

Stiffness matrix K:
[[ 1 -2  0  0  0]
 [-2  5 -3  0  0]
 [ 0 -3 10 -5  0]
 [ 0  0 -5 26 -7]
 [ 0  0  0 -7 49]]

Structure:
  K[0,0] = 1  (top level, one constraint)
  K[1,1] = 5  = 2^2 + 1
  K[0,1] = -2  = -2
  K[2,2] = 10  = 3^2 + 1
  K[1,2] = -3  = -3
  K[3,3] = 26  = 5^2 + 1
  K[2,3] = -5  = -5
  K[4,4] = 49  = 7^2  (bottom level)
  K[3,4] = -7  = -7

Eigenvalues of K: [-2.4577e-16  4.2490e+00  1.0180e+01  2.5563e+01  5.1009e+01]
Rank = 4  (one zero mode)


## 2 — The Zero Mode: Influx

$K$ has rank 4. Its kernel is the state where all covering constraints
are perfectly satisfied: $\theta_{k-1} = p_k\,\theta_k$ for every $k$.

Starting from $\theta_0 = 1$:
$$\theta_k = \frac{\theta_0}{P_k} = \frac{1}{P_k}$$

So the zero mode is $\mathbf{v}_0 = (1/P_0,\; 1/P_1,\; \ldots,\; 1/P_4)
= (1,\; \tfrac{1}{2},\; \tfrac{1}{6},\; \tfrac{1}{30},\; \tfrac{1}{210})$.

This is **influx**: all faculties rotating in perfect sync at rates
determined by their covering factors. The heavier (more folded)
levels move proportionally less — mass arises from multiplicity.

In [3]:
# Zero mode: v0_k = 1 / P_k
v0 = np.array([1.0 / P for P in primorial_seq])
residual = K @ v0

print("Zero mode v0 = 1/P_k:")
for k, (P, v) in enumerate(zip(primorial_seq, v0)):
    print(f"  Level {k}: P_k = {P:>3d},  v0 = 1/{P} = {v:.6f}")
print(f"\nK @ v0 = {residual}")
print(f"Max |residual| = {np.max(np.abs(residual)):.2e}")
assert np.max(np.abs(residual)) < 1e-14, "Zero mode failed!"

# Physical meaning: the constraint p_k * theta_k = theta_{k-1}
# For v0: p_k * (1/P_k) = p_k / (P_{k-1} * p_k) = 1/P_{k-1} = v0_{k-1}
print("\nCovering check: p_k * v0_k == v0_{k-1}")
for i, p in enumerate(primes):
    lhs = p * v0[i + 1]
    rhs = v0[i]
    print(f"  p_{i+1}={p}: {p} * {v0[i+1]:.6f} = {lhs:.6f} == {rhs:.6f}  {'OK' if abs(lhs - rhs) < 1e-14 else 'FAIL'}")

Zero mode v0 = 1/P_k:
  Level 0: P_k =   1,  v0 = 1/1 = 1.000000
  Level 1: P_k =   2,  v0 = 1/2 = 0.500000
  Level 2: P_k =   6,  v0 = 1/6 = 0.166667
  Level 3: P_k =  30,  v0 = 1/30 = 0.033333
  Level 4: P_k = 210,  v0 = 1/210 = 0.004762

K @ v0 = [ 0.00000000e+00  0.00000000e+00 -2.22044605e-16  9.71445147e-17
  2.77555756e-17]
Max |residual| = 2.22e-16

Covering check: p_k * v0_k == v0_{k-1}
  p_1=2: 2 * 0.500000 = 1.000000 == 1.000000  OK
  p_2=3: 3 * 0.166667 = 0.500000 == 0.500000  OK
  p_3=5: 5 * 0.033333 = 0.166667 == 0.166667  OK
  p_4=7: 7 * 0.004762 = 0.033333 == 0.033333  OK


## 3 — Normal Modes

The equations of motion $W\,\ddot{\boldsymbol\theta} = -\kappa\,K\,\boldsymbol\theta$
give the generalized eigenvalue problem

$$K\,\mathbf{v} = \omega^2\,W\,\mathbf{v}.$$

One eigenvalue is zero (the influx mode). The remaining four give
oscillation frequencies $\omega_1 < \omega_2 < \omega_3 < \omega_4$ that are
**pure functions of $\{2,3,5,7\}$** — zero free parameters.

In [4]:
# Mass matrix: primorial weights
W = np.diag([float(w) for w in primorial_seq])

# Generalized eigenvalue problem: K v = omega^2 W v
omega2, modes = eigh(K, W)

print("Generalized eigenvalues (omega^2):")
for i, w2 in enumerate(omega2):
    tag = "  <-- zero mode (influx)" if abs(w2) < 1e-10 else ""
    print(f"  Mode {i}: omega^2 = {w2:>12.6f}{tag}")

nonzero = omega2[omega2 > 1e-10]
freqs = np.sqrt(nonzero)
print(f"\nOscillation frequencies: {np.array2string(freqs, precision=4)}")

mass_ratios = freqs / freqs[0]
print(f"Mass ratios (to lightest): {np.array2string(mass_ratios, precision=3)}")
print(f"Range: 1 : {mass_ratios[-1]:.3f}  (factor of {mass_ratios[-1]:.1f})")

print("\nMode shapes (columns = eigenvectors):")
for i in range(n_levels):
    v = modes[:, i]
    w2 = omega2[i]
    dom = np.argmax(np.abs(v))
    tag = f"  (dominant at level {dom})"
    print(f"  Mode {i} [omega^2={w2:>9.4f}]: [{', '.join(f'{x:+.4f}' for x in v)}]{tag}")

Generalized eigenvalues (omega^2):
  Mode 0: omega^2 =     0.000000  <-- zero mode (influx)
  Mode 1: omega^2 =     0.220621
  Mode 2: omega^2 =     0.748181
  Mode 3: omega^2 =     1.652806
  Mode 4: omega^2 =     3.645058

Oscillation frequencies: [0.4697 0.865  1.2856 1.9092]
Mass ratios (to lightest): [1.    1.842 2.737 4.065]
Range: 1 : 4.065  (factor of 4.1)

Mode shapes (columns = eigenvectors):
  Mode 0 [omega^2=   0.0000]: [-0.7659, -0.3829, -0.1276, -0.0255, -0.0036]  (dominant at level 0)
  Mode 1 [omega^2=   0.2206]: [-0.0715, -0.0279, +0.0053, +0.0260, +0.0681]  (dominant at level 0)
  Mode 2 [omega^2=   0.7482]: [+0.2530, +0.0318, -0.1314, -0.1640, +0.0106]  (dominant at level 0)
  Mode 3 [omega^2=   1.6528]: [+0.3907, -0.1275, -0.3325, +0.0710, -0.0017]  (dominant at level 0)
  Mode 4 [omega^2=   3.6451]: [-0.4378, +0.5791, -0.1501, +0.0090, -0.0001]  (dominant at level 1)


## 4 — Exact Rational Invariants

The trace and product of $\omega^2$ are invariants of the characteristic
polynomial of $W^{-1}K$. Since both $K$ and $W$ have integer entries,
these invariants must be **exact rationals**.

$$\sum_{i} \omega_i^2 \;=\; \mathrm{tr}(W^{-1}K),
\qquad
\prod_{i \ne 0} \omega_i^2 \;=\; s_4\bigl(\mathrm{char.\;poly.}\bigr)$$

where $s_4$ is the coefficient of $\lambda$ in the degree-5 characteristic
polynomial (the elementary symmetric polynomial of the nonzero eigenvalues).

In [5]:
# Build W^{-1}K with exact Fraction arithmetic
n = n_levels
K_exact = [[Fraction(0)] * n for _ in range(n)]
for i, p in enumerate(primes):
    lo, hi = i, i + 1
    K_exact[lo][lo] += Fraction(1)
    K_exact[hi][hi] += Fraction(p**2)
    K_exact[lo][hi] -= Fraction(p)
    K_exact[hi][lo] -= Fraction(p)

W_inv = [Fraction(1, primorial_seq[k]) for k in range(n)]
WinvK = [[W_inv[r] * K_exact[r][c] for c in range(n)] for r in range(n)]

# Trace = sum of diagonal of W^{-1}K
trace_exact = sum(WinvK[k][k] for k in range(n))
print(f"tr(W^-1 K) = {trace_exact} = {float(trace_exact):.10f}")

# For the product: compute cofactors (4x4 principal minors of W^{-1}K)
# s_4 = sum of all 4x4 principal minors
def det_fraction(M):
    n = len(M)
    if n == 1:
        return M[0][0]
    if n == 2:
        return M[0][0] * M[1][1] - M[0][1] * M[1][0]
    total = Fraction(0)
    for j in range(n):
        sub = [[M[r][c] for c in range(n) if c != j] for r in range(1, n)]
        sign = Fraction(1) if j % 2 == 0 else Fraction(-1)
        total += sign * M[0][j] * det_fraction(sub)
    return total

# Sum of 4x4 principal minors (exclude each row/col in turn)
s4 = Fraction(0)
minor_values = []
for excl in range(n):
    rows = [r for r in range(n) if r != excl]
    sub = [[WinvK[r][c] for c in rows] for r in rows]
    m = det_fraction(sub)
    minor_values.append(m)
    s4 += m

print(f"\nCofactors (4x4 principal minors of W^-1 K):")
for k, m in enumerate(minor_values):
    print(f"  Excluding level {k}: {m} = {float(m):.10f}")
print(f"\ns_4 = sum of cofactors = {s4} = {float(s4):.10f}")

# Verify
print(f"\nProduct of nonzero omega^2:")
print(f"  Exact:     {s4} = {float(s4):.10f}")
print(f"  Numerical: {np.prod(nonzero):.10f}")
print(f"  Match: {abs(float(s4) - np.prod(nonzero)) < 1e-12}")

print(f"\n  === IDENTITY 42 ===")
print(f"  Sum   omega^2 = {trace_exact}  = {float(trace_exact)}")
print(f"  Prod  omega^2 = {s4}")
print(f"  Both exact rationals. Zero free parameters.")

tr(W^-1 K) = 94/15 = 6.2666666667

Cofactors (4x4 principal minors of W^-1 K):
  Excluding level 0: 7/12 = 0.5833333333
  Excluding level 1: 7/24 = 0.2916666667
  Excluding level 2: 7/72 = 0.0972222222
  Excluding level 3: 7/360 = 0.0194444444
  Excluding level 4: 1/360 = 0.0027777778

s_4 = sum of cofactors = 179/180 = 0.9944444444

Product of nonzero omega^2:
  Exact:     179/180 = 0.9944444444
  Numerical: 0.9944444444
  Match: True

  === IDENTITY 42 ===
  Sum   omega^2 = 94/15  = 6.266666666666667
  Prod  omega^2 = 179/180
  Both exact rationals. Zero free parameters.


## 5 — The Graph Laplacian on $\mathbb{Z}^*_{210}$

The discrete state space $\mathbb{Z}^*_{210}$ has 48 elements. A generating
set $S$ defines a Cayley graph: nodes are group elements, edges connect
$g$ to $g \cdot s$ for each $s \in S \cup S^{-1}$.

The graph Laplacian $L = D - A$ encodes the discrete geometry.
Its **spectral gap** $\lambda_1$ (smallest nonzero eigenvalue)
is the mass gap of the discrete system.

**Key question:** does the choice of generators affect the geometry?

In [6]:
def build_cayley_laplacian(gens):
    n = PHI
    A = np.zeros((n, n))
    for g in gens:
        g_inv = SA.inverse(g)
        for k in SA.Z_star:
            # Forward: k -> k*g
            i, j = SA._idx[k], SA._idx[SA.multiply(k, g)]
            A[i, j] = 1
            A[j, i] = 1
            # Inverse: k -> k*g^{-1} (only if distinct)
            if g_inv != g:
                i2, j2 = SA._idx[k], SA._idx[SA.multiply(k, g_inv)]
                A[i2, j2] = 1
                A[j2, i2] = 1
    D = np.diag(A.sum(axis=1))
    return D - A, A

# Per-prime generators: each acts on EXACTLY one prime factor
# Z*_2 is trivial (only element is 1), so skip it
per_prime_gens = {}
for p in primes:
    if p == 2:
        continue
    prim_root = SA.primitive_roots[p]
    residues = [1 if q != p else prim_root for q in primes]
    gen = SA.reconstruct(residues)
    per_prime_gens[p] = gen

natural_gens = list(per_prime_gens.values())
print("Per-prime generators (each acts on one faculty):")
for p, g in per_prime_gens.items():
    print(f"  Prime {p}: g = {g},  decomp = {SA.decompose(g)},  order = {SA.order(g)}")
print(f"Generates Z*_210: {SA.generates_group(natural_gens)}")

L_sep, A_sep = build_cayley_laplacian(natural_gens)
evals_sep = np.sort(np.linalg.eigvalsh(L_sep))
degree_sep = int(A_sep.sum(axis=1)[0])

gap_sep = evals_sep[evals_sep > 1e-10][0]
print(f"\nCayley graph (per-prime generators):")
print(f"  Degree per node: {degree_sep}")
print(f"  Spectral gap:    {gap_sep:.6f}")
print(f"  Max eigenvalue:  {evals_sep[-1]:.6f}")

Per-prime generators (each acts on one faculty):
  Prime 3: g = 71,  decomp = (1, 2, 1, 1),  order = 2
  Prime 5: g = 127,  decomp = (1, 1, 2, 1),  order = 4
  Prime 7: g = 31,  decomp = (1, 1, 1, 3),  order = 6
Generates Z*_210: True

Cayley graph (per-prime generators):
  Degree per node: 5
  Spectral gap:    1.000000
  Max eigenvalue:  10.000000


## 6 — Laplacian Factoring: The CRT in Geometry

Since each per-prime generator acts on exactly one cyclic factor,
the Cayley graph is a **Cartesian product** of per-prime Cayley graphs.
The Laplacian factors as

$$L \;=\; L_3 \otimes I_5 \otimes I_7 \;+\; I_3 \otimes L_5 \otimes I_7
\;+\; I_3 \otimes I_5 \otimes L_7$$

and the spectrum is all sums $\lambda_3 + \lambda_5 + \lambda_7$ where
$\lambda_p$ ranges over $\operatorname{spec}(L_p)$.

**Subtlety:** For $\mathbb{Z}^*_3 = \{1,2\}$, the generator $2$ is
**self-inverse** ($2^2 \equiv 1 \pmod{3}$), giving degree 1 and
eigenvalues $\{0, 2\}$. For $\mathbb{Z}^*_5$ and $\mathbb{Z}^*_7$,
generator and inverse are distinct, giving degree 2 and the standard
cycle eigenvalues $2 - 2\cos(2\pi k / n)$.

In [7]:
# Per-prime Laplacian eigenvalues (CORRECTED for self-inverse case)
per_prime_spectra = {}
per_prime_degree = {}
for p in primes:
    if p == 2:
        per_prime_spectra[2] = [0.0]
        per_prime_degree[2] = 0
        continue
    phi_p = p - 1
    g_p = SA.primitive_roots[p]
    g_inv = pow(g_p, -1, p)
    if g_p == g_inv:
        # Self-inverse generator: degree 1 Cayley graph
        evals_p = [1 - np.cos(2 * np.pi * k / phi_p) for k in range(phi_p)]
        per_prime_degree[p] = 1
    else:
        # Generator + distinct inverse: degree 2 cycle
        evals_p = [2 - 2 * np.cos(2 * np.pi * k / phi_p) for k in range(phi_p)]
        per_prime_degree[p] = 2
    per_prime_spectra[p] = sorted(evals_p)

print("Per-prime Cayley graph spectra:")
for p in primes:
    if p == 2:
        continue
    g_p = SA.primitive_roots[p]
    g_inv = pow(g_p, -1, p)
    self_inv = " (self-inverse)" if g_p == g_inv else ""
    print(f"  Z*_{p}: gen={g_p}, inv={g_inv}{self_inv}, "
          f"degree={per_prime_degree[p]}, "
          f"spec={[round(e, 4) for e in per_prime_spectra[p]]}")

total_degree = sum(per_prime_degree.values())
print(f"\nTotal degree = {' + '.join(str(per_prime_degree[p]) for p in primes)} = {total_degree}")
print(f"Matches Cayley graph degree: {total_degree == degree_sep}")

# Predicted spectrum: all sums of per-prime eigenvalues
from itertools import product as cartesian_product
predicted = sorted(
    l3 + l5 + l7
    for l3 in per_prime_spectra[3]
    for l5 in per_prime_spectra[5]
    for l7 in per_prime_spectra[7]
)

max_diff = max(abs(p - a) for p, a in zip(predicted, evals_sep))
print(f"\nPredicted vs actual spectrum:")
print(f"  Count: predicted={len(predicted)}, actual={len(evals_sep)} (both = phi(210)={PHI})")
print(f"  Max difference: {max_diff:.2e}")

if max_diff < 1e-10:
    print("  EXACT MATCH: L factors as tensor sum of per-prime Laplacians.")
else:
    print("  MISMATCH detected.")
    print("  First 12 eigenvalues:")
    for i in range(12):
        match = abs(predicted[i] - evals_sep[i]) < 1e-10
        print(f"    pred={predicted[i]:.6f}  actual={evals_sep[i]:.6f}  {'OK' if match else 'DIFF'}")

Per-prime Cayley graph spectra:
  Z*_3: gen=2, inv=2 (self-inverse), degree=1, spec=[np.float64(0.0), np.float64(2.0)]
  Z*_5: gen=2, inv=3, degree=2, spec=[np.float64(0.0), np.float64(2.0), np.float64(2.0), np.float64(4.0)]
  Z*_7: gen=3, inv=5, degree=2, spec=[np.float64(0.0), np.float64(1.0), np.float64(1.0), np.float64(3.0), np.float64(3.0), np.float64(4.0)]

Total degree = 0 + 1 + 2 + 2 = 5
Matches Cayley graph degree: True

Predicted vs actual spectrum:
  Count: predicted=48, actual=48 (both = phi(210)=48)
  Max difference: 7.11e-15
  EXACT MATCH: L factors as tensor sum of per-prime Laplacians.


## 7 — Two Routes to Coupling

The per-prime generators restore CRT separability in the discrete
geometry: the Laplacian factors exactly. But what if we use generators
that act on **multiple primes simultaneously**?

Max-order generators (order 12 = $\lambda(210)$) have nontrivial
components in every cyclic factor. They couple the faculties.

| Route | Mechanism | Coupling |
|-------|-----------|----------|
| **Continuous** | Covering constraint $\theta_{k-1} = p_k\theta_k$ | Tridiagonal $K$ |
| **Discrete** | Multi-prime generators | Non-separable $L$ |

Both break algebraic independence through **geometry**.

In [8]:
# Find generating triple from max-order elements
max_gens = SA.max_order_generators
found = None
for i, g1 in enumerate(max_gens):
    for j, g2 in enumerate(max_gens[i+1:], i+1):
        for g3 in max_gens[j+1:]:
            if SA.generates_group([g1, g2, g3]):
                found = [g1, g2, g3]
                break
        if found:
            break
    if found:
        break

print(f"Max-order generating set: {found}")
print(f"Decompositions: {[SA.decompose(g) for g in found]}")

L_coupled, A_coupled = build_cayley_laplacian(found)
evals_coupled = np.sort(np.linalg.eigvalsh(L_coupled))
degree_coupled = int(A_coupled.sum(axis=1)[0])

gap_coupled = evals_coupled[evals_coupled > 1e-10][0]
print(f"\nCayley graph (max-order generators):")
print(f"  Degree per node: {degree_coupled}")
print(f"  Spectral gap:    {gap_coupled:.6f}")
print(f"  Max eigenvalue:  {evals_coupled[-1]:.6f}")

# Compare: which eigenvalues are NEW?
print(f"\nFirst 12 eigenvalues:")
print(f"  {'Separable':>10s}  {'Coupled':>10s}  Note")
for i in range(12):
    e_sep = evals_sep[i]
    e_coup = evals_coupled[i]
    in_sep = any(abs(e_coup - p) < 1e-6 for p in predicted)
    note = "" if in_sep else " <-- NEW"
    print(f"  {e_sep:>10.6f}  {e_coup:>10.6f}{note}")

print(f"\n  Separable gap:  {gap_sep:.6f}  (exact integer = 1)")
print(f"  Coupled gap:    {gap_coupled:.6f}  (smaller: coupling slows mixing)")
print(f"  Gap ratio:      {gap_coupled / gap_sep:.4f}")

Max-order generating set: [17, 23, 37]
Decompositions: [(1, 2, 2, 3), (1, 2, 3, 2), (1, 1, 2, 2)]

Cayley graph (max-order generators):
  Degree per node: 6
  Spectral gap:    0.803848
  Max eigenvalue:  12.000000

First 12 eigenvalues:
   Separable     Coupled  Note
    0.000000    0.000000
    1.000000    0.803848 <-- NEW
    1.000000    0.803848 <-- NEW
    2.000000    3.000000
    2.000000    3.000000
    2.000000    4.000000
    3.000000    4.000000
    3.000000    4.000000
    3.000000    4.267949 <-- NEW
    3.000000    4.267949 <-- NEW
    3.000000    4.267949 <-- NEW
    3.000000    4.267949 <-- NEW

  Separable gap:  1.000000  (exact integer = 1)
  Coupled gap:    0.803848  (smaller: coupling slows mixing)
  Gap ratio:      0.8038


## 8 — Curvature Hierarchy

On $S^2 \times \mathbb{R}^+$, each prime orbit lives at radius $P_k$
(the primorial). The Gaussian curvature of a sphere of radius $r$ is
$\kappa = 1/r^2$, giving

$$\kappa_k = \frac{1}{P_k^2}, \qquad
\frac{\kappa_k}{\kappa_{k+1}} = \frac{P_{k+1}^2}{P_k^2}
= p_{k+1}^2.$$

Curvature drops by the **square of each prime** between levels.
The total range is $(P_4/P_1)^2 = 105^2 = 11{,}025$.

The inter-level curvature difference $\Delta\kappa_k = \kappa_k - \kappa_{k+1}$
gives the **gravitational coupling** between adjacent nesting levels.

In [9]:
print("Curvature hierarchy (kappa_k = 1/P_k^2):\n")
header = f"  {'Level':>5s}  {'P_k':>5s}  {'kappa':>14s}  {'ratio to prev':>14s}"
print(header)
print("  " + "-" * len(header.strip()))
prev_kappa = None
for k, Pk in enumerate(PRIMORIALS):
    kappa = Fraction(1, Pk**2)
    ratio_str = ""
    if prev_kappa is not None:
        ratio = prev_kappa / kappa
        ratio_str = f"{ratio} = {primes[k]}^2"
    print(f"  {k+1:>5d}  {Pk:>5d}  1/{Pk**2:<13d}  {ratio_str}")
    prev_kappa = kappa

total_range = PRIMORIALS[-1]**2 // PRIMORIALS[0]**2
print(f"\n  Total range: kappa_1/kappa_4 = {total_range} = {PRIMORIALS[-1]//PRIMORIALS[0]}^2")

print(f"\nInter-level gravitational coupling (Delta kappa):")
for k in range(len(PRIMORIALS) - 1):
    dk = Fraction(1, PRIMORIALS[k]**2) - Fraction(1, PRIMORIALS[k+1]**2)
    print(f"  Levels {k+1} <-> {k+2}: Delta_kappa = {dk} = {float(dk):.6e}")
total_dk = Fraction(1, PRIMORIALS[0]**2) - Fraction(1, PRIMORIALS[-1]**2)
print(f"  Total (1 <-> 4):  Delta_kappa = {total_dk} = {float(total_dk):.6e}")

Curvature hierarchy (kappa_k = 1/P_k^2):

  Level    P_k           kappa   ratio to prev
  --------------------------------------------
      1      2  1/4              
      2      6  1/36             9 = 3^2
      3     30  1/900            25 = 5^2
      4    210  1/44100          49 = 7^2

  Total range: kappa_1/kappa_4 = 11025 = 105^2

Inter-level gravitational coupling (Delta kappa):
  Levels 1 <-> 2: Delta_kappa = 2/9 = 2.222222e-01
  Levels 2 <-> 3: Delta_kappa = 2/75 = 2.666667e-02
  Levels 3 <-> 4: Delta_kappa = 4/3675 = 1.088435e-03
  Total (1 <-> 4):  Delta_kappa = 2756/11025 = 2.499773e-01


## 9 — Scorecard

| # | Identity | Source | Free parameters |
|---|----------|--------|-----------------|
| **41** | $K$ is tridiagonal with $\ker(K) = \text{span}(1/P_k)$ | Covering constraint | 0 |
| **42** | $\sum\omega^2 = 94/15$, $\prod\omega^2 = 179/180$ | Characteristic polynomial | 0 |
| **43** | Cayley Laplacian factors exactly under per-prime generators; $\lambda_1 = 1$ | CRT + cycle Laplacian | 0 |
| **44** | Multi-prime generators break factoring; spectral gap drops | Non-separable Cayley graph | 0 |
| **45** | $\kappa_k = 1/P_k^2$, dropping by $p_{k+1}^2$ between levels | Spherical embedding | 0 |

**Running total: 45 structural identities, 0 free parameters.**

**Null Finding:** Normal mode mass ratios ($1 : 1.84 : 2.74 : 4.07$) span
a factor of $\sim\!4$. Lepton mass ratios span $\sim\!3500$.
The solenoid-Lagrangian oscillation modes describe *inter-level dynamics*,
not particle masses. The hierarchy problem requires a different channel —
perhaps the character-theoretic structure of $\mathbb{Z}^*_{210}$.

In [10]:
# Scorecard verification
print("=" * 60)
print("SCORECARD VERIFICATION")
print("=" * 60)

checks = {}

# Identity 41: K is tridiagonal, rank 4, kernel = (1/P_k)
checks["41: K tridiagonal, rank 4"] = (
    np.sum(np.abs(K) > 1e-10) == 3 * n_levels - 2  # tridiagonal nonzeros
    and np.sum(np.linalg.eigvalsh(K) > 1e-10) == 4   # rank 4
    and np.max(np.abs(K @ v0)) < 1e-14                # kernel correct
)

# Identity 42: exact invariants
checks["42: sum=94/15, prod=179/180"] = (
    trace_exact == Fraction(94, 15)
    and s4 == Fraction(179, 180)
)

# Identity 43: Laplacian factors, gap = 1
checks["43: L factors, gap = 1"] = (
    max_diff < 1e-10
    and abs(gap_sep - 1.0) < 1e-10
)

# Identity 44: coupled generators break factoring
checks["44: coupling breaks factoring"] = (
    gap_coupled < gap_sep - 0.1  # gap strictly smaller
)

# Identity 45: curvature hierarchy
kappa_ratios = [
    PRIMORIALS[k+1]**2 / PRIMORIALS[k]**2
    for k in range(len(PRIMORIALS) - 1)
]
expected_ratios = [p**2 for p in primes[1:]]  # [9, 25, 49]
checks["45: kappa drops by p_{k+1}^2"] = (
    kappa_ratios == expected_ratios
)

for label, ok in checks.items():
    status = "PASS" if ok else "FAIL"
    print(f"  [{status}] Identity {label}")

n_pass = sum(checks.values())
print(f"\n  {n_pass}/{len(checks)} identities verified.")
print(f"  Running total: 45 structural identities, 0 free parameters.")

SCORECARD VERIFICATION
  [PASS] Identity 41: K tridiagonal, rank 4
  [PASS] Identity 42: sum=94/15, prod=179/180
  [PASS] Identity 43: L factors, gap = 1
  [PASS] Identity 44: coupling breaks factoring
  [PASS] Identity 45: kappa drops by p_{k+1}^2

  5/5 identities verified.
  Running total: 45 structural identities, 0 free parameters.


## Summary and Frontier

**What this notebook established:**

1. The covering constraints create a **tridiagonal stiffness matrix** $K$
   with exactly one zero mode: the state of perfect inter-level alignment
   (influx). The four oscillation modes are deviations from this alignment.

2. Both the oscillation invariants ($\sum\omega^2 = 94/15$,
   $\prod\omega^2 = 179/180$) are **exact rationals** — no approximation,
   no fitting, no free parameters.

3. **Two routes to coupling exist:**
   - *Continuous:* covering constraints $\to$ tridiagonal $K$
   - *Discrete:* multi-prime generators $\to$ non-separable Cayley graph
   - Both break CRT separability through geometry.

4. The curvature hierarchy $\kappa_k = 1/P_k^2$ drops by $p_{k+1}^2$
   between levels, giving an 11,025:1 total range from four primes.

5. **Null finding:** normal mode mass ratios ($\sim\!4:1$ range) do not
   match particle mass hierarchies ($\sim\!3500:1$). The solenoid-Lagrangian
   modes are inter-level dynamics, not particle physics.

**Next frontiers:**
- Compute the solenoid metric and geodesics explicitly
- Extract a gravitational coupling constant from the curvature gradient
- Investigate the character-theoretic channel for exponential mass gaps
- Relate the product $179/180$ to the prime structure analytically